# Notebook 05 — Hybrid Retrieval Agent

**Purpose**: Implement **Agent 3** — the Hybrid Retrieval Agent that combines dense vector search, BM25, collaborative filtering (ALS), and citation-graph centrality to produce a personalized Top-K candidate paper set (with CRAG confidence labelling and optional external fallback).

**Sources**:
1. Dense retrieval — **GTE-Large** embeddings, **direct cosine** over the in-memory matrix (no ChromaDB); the query encoder is matched to the doc embeddings
2. BM25 (Okapi)
3. Collaborative filtering — implicit-feedback ALS (Hu, Koren & Volinsky, 2008), one model trained once in `10_recommendation_quality_eval` and shared across notebooks (`als_item_factors.npy` / `als_user_factors.npy` / `als_user_ids.json`)
4. Citation-graph centrality — S2AG `citationCount`, log-scaled (the graph weight `lambda3`)
5. *(optional, off by default)* Semantic Scholar paper search + Tavily web fallback — external ids, augmentation only

All four scored signals (dense, BM25, CF, graph) are min-max normalized across each query's candidate set before being combined with fixed weights (`lambda1..4`) — a **weighted hybridization** design (Burke, 2002), chosen deliberately over a learned/gated fusion because this paper's contribution is citation-graph-grounded relational verification (SciRecTrace), not the retrieval fusion architecture itself. Weights were tuned on a held-out validation split in `10_recommendation_quality_eval`, not picked arbitrarily.

**Outputs**: Top-K `CandidatePaper` objects with source provenance and multi-factor relevance scores.

In [ ]:
# == Dependencies + environment ==============================================
import json, os, pickle, time, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

import importlib.util as _ilu, subprocess as _sp, sys as _sys
for _pkg in ["rank-bm25", "sentence-transformers"]:
    if _ilu.find_spec(_pkg.replace("-", "_")) is None:
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

def _find(name):
    for root in ["/kaggle/input", "/kaggle/working", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    return None
# NOTE: plain os.path.exists("/kaggle") is not reliable on Windows -- it matches
# case-insensitively against any local "kaggle"-named folder on the current drive root.
# Require the actual Kaggle run-type env var, which only a real Kaggle kernel sets.
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
OUT_DIR   = Path("/kaggle/working") if IS_KAGGLE else Path("../data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

S2_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    S2_API_KEY = UserSecretsClient().get_secret("SEMANTIC_SCHOLAR_API_KEY") or S2_API_KEY
except Exception:
    pass
TAVILY_KEY = os.getenv("TAVILY_API_KEY", "")
HEADERS_S2 = {"x-api-key": S2_API_KEY} if S2_API_KEY else {}
S2_BASE    = "https://api.semanticscholar.org/graph/v1"
print("S2 key:", bool(S2_API_KEY), "| Tavily:", bool(TAVILY_KEY))

In [ ]:
# == Load artifacts (same dataset paths as nb04; reuse existing GTE embeddings) =
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Papers metadata: same source as nb04 (JSON at reapers2025), _find fallback
_PAPERS_JSON = Path("/kaggle/input/datasets/reapers2025/paper-metadata/paper_metadata.json")
if _PAPERS_JSON.exists():
    papers_df = pd.read_json(_PAPERS_JSON)
else:
    _pp = _find("paper_metadata.json") or _find("papers_metadata.csv")
    assert _pp is not None, "papers metadata not found (paper_metadata.json / papers_metadata.csv)"
    papers_df = pd.read_json(_pp) if str(_pp).endswith(".json") else pd.read_csv(_pp)
id_col = next((c for c in ["arxiv_id", "paper_id", "paperId", "id"] if c in papers_df.columns), None)

# ---- Embeddings: e5 (nb03) if present, else the existing GTE-Large -----------
# The query encoder MUST match the model that produced the doc embeddings.
_e5_emb, _e5_map = _find("paper_embeddings_e5.npy"), _find("paper_id_mapping_e5.json")
if _e5_emb and _e5_map:
    emb_matrix = np.load(_e5_emb); id_mapping = json.load(open(_e5_map, encoding="utf-8"))
    EMBED_MODEL, QUERY_PREFIX = "intfloat/e5-large-v2", "query: "
else:
    _EMB_HARD = Path("/kaggle/input/datasets/reapers2025/acl-embeddings/paper_embeddings_full.npy")
    _MAP_HARD = Path("/kaggle/input/datasets/reapers2025/acl-embeddings/paper_id_mapping.json")
    _emb_p = _EMB_HARD if _EMB_HARD.exists() else _find("paper_embeddings_full.npy")
    _map_p = _MAP_HARD if _MAP_HARD.exists() else _find("paper_id_mapping.json")
    assert _emb_p and _map_p, "GTE embeddings not found (paper_embeddings_full.npy / paper_id_mapping.json)"
    emb_matrix = np.load(_emb_p); id_mapping = json.load(open(_map_p, encoding="utf-8"))
    EMBED_MODEL, QUERY_PREFIX = "thenlper/gte-large", ""        # GTE: no prefix
emb_matrix  = emb_matrix.astype(np.float32)
_idx_to_pid = {int(v): k for k, v in id_mapping.items()}
_doc_norms  = np.linalg.norm(emb_matrix, axis=1) + 1e-9
embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"embeddings {emb_matrix.shape} | encoder {EMBED_MODEL} | papers {len(papers_df)} (id_col={id_col})")

# ---- BM25: load nb03/nb05 index if present, else build (cached to working) ---
_bm = _find("bm25_index.pkl")
if _bm:
    _d = pickle.load(open(_bm, "rb")); bm25_index = _d["bm25"]; bm25_doc_ids = _d["doc_ids"]
    print("loaded BM25 index")
else:
    print("building BM25 over corpus (one-time, ~1-2 min)...")
    _toks, bm25_doc_ids = [], []
    for _, row in papers_df.iterrows():
        _toks.append((str(row.get("title", "") or "") + " " + str(row.get("abstract", "") or "")).lower().split())
        bm25_doc_ids.append(str(row[id_col]))
    bm25_index = BM25Okapi(_toks)
    with open(OUT_DIR / "bm25_index.pkl", "wb") as f:
        pickle.dump({"bm25": bm25_index, "doc_ids": bm25_doc_ids}, f)
    print("BM25 built + cached ->", OUT_DIR / "bm25_index.pkl")

# ---- Optional artifacts (graceful if absent): graph scores, user profiles (nb04)
_gp = _find("graph_scores.json");  graph_scores  = json.load(open(_gp, encoding="utf-8")) if _gp else {}
_UP_HARD = Path("/kaggle/input/datasets/joytuntonny/rationale-subset/user_profiles.json")
_up = _UP_HARD if _UP_HARD.exists() else _find("user_profiles.json")
user_profiles = json.load(open(_up, encoding="utf-8")) if _up else {}
if not user_profiles:
    print("WARNING: user_profiles.json NOT found - attach nb04's output dataset for personalization. "
          "Running WITHOUT profile reranking.")
print(f"papers {len(papers_df)} | graph_scores {len(graph_scores)} | user_profiles {len(user_profiles)}")

# ---- Collaborative-filtering signal: SHARED ALS model trained once in nb10 ---------------
# One trained model, reused everywhere (nb05/nb07/nb10) rather than retrained per notebook.
# item_factors rows are aligned to id_mapping's insertion order (verified identical convention
# in nb10), so cf_item_factors[id_mapping[pid]] is that paper's latent vector directly --
# no separate item lookup table needed. user_factors rows are aligned to als_user_ids.json;
# a user_id absent from that list is CF-cold-start (no vector -- retrieval falls back to
# content+graph only for that user, same spirit as the base paper's RAG cold-start fallback).
_cf_items_p = _find("als_item_factors.npy")
_cf_users_p = _find("als_user_factors.npy")
_cf_ids_p   = _find("als_user_ids.json")
if _cf_items_p and _cf_users_p and _cf_ids_p:
    cf_item_factors = np.load(_cf_items_p)
    cf_user_factors = np.load(_cf_users_p)
    cf_user_row = {u: i for i, u in enumerate(json.load(open(_cf_ids_p, encoding="utf-8")))}
    print(f"loaded shared ALS CF model: items {cf_item_factors.shape}, users {cf_user_factors.shape} "
          f"({len(cf_user_row)} users with a CF vector)")
else:
    cf_item_factors, cf_user_factors, cf_user_row = None, None, {}
    print("WARNING: shared ALS CF artifacts NOT found (als_item_factors.npy / als_user_factors.npy / "
          "als_user_ids.json) -- attach nb10's output dataset. Running WITHOUT the CF signal "
          "(retrieval falls back to dense+BM25+graph only, as before).")

In [ ]:
# == Graph-augmented ranking: citation-graph centrality (S2AG citation counts) ==
# Re-ranks candidates by citation-graph importance (log-scaled to dampen popularity bias),
# cached to working. A learned GraphSAGE GNN needs the full corpus citation graph (infeasible
# to fetch wholesale), so citation centrality is the practical, robust graph-augmented signal.
import re, math, requests
_GS_CACHE = OUT_DIR / "graph_scores.json"
if not graph_scores and _GS_CACHE.exists():
    graph_scores = json.load(open(_GS_CACHE, encoding="utf-8"))
    print(f"loaded cached graph_scores ({len(graph_scores)})")
elif not graph_scores:
    def _s2id(x): return ("ARXIV:" + str(x)) if re.match(r"^\d{4}\.\d{4,5}", str(x)) else str(x)
    ids = [str(r[id_col]) for _, r in papers_df.iterrows() if r.get(id_col) is not None]
    print(f"fetching citation counts for {len(ids)} papers via S2 batch (one-time, ~15-20 min, cached)...")
    raw, CH = {}, 500
    for s in range(0, len(ids), CH):
        sub = ids[s:s + CH]
        for attempt in range(5):
            try:
                rr = requests.post(f"{S2_BASE}/paper/batch", headers=HEADERS_S2,
                                   params={"fields": "citationCount"},
                                   json={"ids": [_s2id(x) for x in sub]}, timeout=40)
                if rr.status_code == 200:
                    for pid, obj in zip(sub, rr.json()):
                        if obj and obj.get("citationCount") is not None:
                            raw[pid] = obj["citationCount"]
                    break
                elif rr.status_code == 429:
                    time.sleep(2 * (attempt + 1))
                else:
                    time.sleep(1); break
            except Exception:
                time.sleep(1)
        time.sleep(0.3)
        if s and s % 10000 == 0:
            print(f"  {s}/{len(ids)}")
    if raw:
        lg = {k: math.log1p(v) for k, v in raw.items()}
        lo, hi = min(lg.values()), max(lg.values())
        graph_scores = {k: ((v - lo) / (hi - lo) if hi > lo else 0.0) for k, v in lg.items()}
    json.dump(graph_scores, open(_GS_CACHE, "w", encoding="utf-8"))
    print(f"built graph_scores for {len(graph_scores)} papers -> {_GS_CACHE}")
print(f"graph_scores ready: {len(graph_scores)} papers (used via the agent's lambda3 graph weight)")

## 1. Candidate Paper Dataclass

In [ ]:
@dataclass
class CandidatePaper:
    paper_id: str
    title: str = ""
    abstract: str = ""
    year: Optional[int] = None
    venue: str = ""
    authors: str = ""
    # Retrieval scores
    dense_score: float = 0.0
    bm25_score: float = 0.0
    graph_score: float = 0.0
    cf_score: float = 0.0
    final_score: float = 0.0
    # Provenance
    sources: List[str] = field(default_factory=list)  # ["dense", "bm25", "s2api", "web"]
    retrieval_confidence: str = "CORRECT"  # CORRECT / AMBIGUOUS / INCORRECT (CRAG)

print("CandidatePaper defined.")

## 2. Retrieval Sub-modules

In [ ]:
def dense_retrieve(query_text: str, top_k: int = 20) -> List[Tuple[str, float]]:
    """Dense retrieval via direct cosine over the embedding matrix (no ChromaDB)."""
    q = embed_model.encode([QUERY_PREFIX + query_text], normalize_embeddings=True,
                           convert_to_numpy=True)[0].astype(np.float32)
    sims = (emb_matrix @ q) / _doc_norms          # cosine (q is unit-norm)
    k = min(top_k, len(sims))
    top = np.argpartition(-sims, k - 1)[:k]
    top = top[np.argsort(-sims[top])]
    return [(_idx_to_pid.get(int(i), str(i)), float(sims[i])) for i in top]


def bm25_retrieve(query_text: str, top_k: int = 20) -> List[Tuple[str, float]]:
    """BM25 retrieval."""
    tokens = query_text.lower().split()
    scores = bm25_index.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    max_score = scores[top_idx[0]] if scores[top_idx[0]] > 0 else 1.0
    return [
        (bm25_doc_ids[i], float(scores[i] / max_score))
        for i in top_idx if i < len(bm25_doc_ids) and scores[i] > 0
    ]


def s2_retrieve(query_text: str, top_k: int = 20) -> List[Tuple[str, float]]:
    """Semantic Scholar API retrieval."""
    if not S2_API_KEY:
        return []
    import requests
    url = f"{S2_BASE}/paper/search"
    params = {"query": query_text, "limit": top_k,
              "fields": "paperId,title,abstract,year,venue",
              "fieldsOfStudy": "Computer Science"}
    try:
        resp = requests.get(url, headers=HEADERS_S2, params=params, timeout=15)
        if resp.status_code == 200:
            papers = resp.json().get("data", [])
            # Score by rank position (proxy for relevance)
            return [(p["paperId"], 1.0 - i * 0.04) for i, p in enumerate(papers) if p.get("paperId")]
    except Exception:
        pass
    return []


def tavily_retrieve(query_text: str, top_k: int = 5) -> List[Tuple[str, float]]:
    """Tavily web search fallback (CRAG trigger)."""
    if not TAVILY_KEY:
        return []
    try:
        from tavily import TavilyClient
        client = TavilyClient(api_key=TAVILY_KEY)
        results = client.search(
            query=f"scientific paper {query_text} arxiv",
            search_depth="basic", max_results=top_k,
            include_domains=["arxiv.org", "aclanthology.org", "semanticscholar.org"],
        )
        # Return URL-based IDs (will need resolution in CRAG evaluator)
        return [(r["url"], r.get("score", 0.5)) for r in results.get("results", [])]
    except Exception:
        pass
    return []

print("Retrieval sub-modules defined.")

## 3. CRAG Retrieval Evaluator

In [ ]:
CRAG_THRESHOLD_CORRECT   = 0.75
CRAG_THRESHOLD_INCORRECT = 0.35

def crag_evaluate(paper_id: str, query_text: str, dense_score: float) -> str:
    """
    CRAG retrieval evaluator: classifies a retrieved paper as
    CORRECT / AMBIGUOUS / INCORRECT based on dense similarity score.
    In production, this should use a cross-encoder re-ranker.
    """
    if dense_score >= CRAG_THRESHOLD_CORRECT:
        return "CORRECT"
    elif dense_score <= CRAG_THRESHOLD_INCORRECT:
        return "INCORRECT"
    else:
        return "AMBIGUOUS"


def decompose_recompose(paper_id: str, abstract: str) -> str:
    """
    CRAG: decompose AMBIGUOUS abstract into clean segments,
    re-score, and return the most relevant portion.
    """
    sentences = abstract.split(". ")
    # In production: re-rank sentences with cross-encoder against query
    # Here: return first 3 sentences as cleaned context
    return ". ".join(sentences[:3])

print("CRAG evaluator defined.")

## 4. Hybrid Retrieval Agent — Main Class

In [ ]:
class HybridRetrievalAgent:
    """
    Agent 3: Multi-source hybrid retrieval -- dense + BM25 + collaborative filtering (ALS)
    + citation-graph centrality -- with CRAG fallback for low-confidence retrievals.

    Weighted hybridization (Burke, 2002) rather than a learned fusion: this paper's
    contribution is citation-graph-grounded relational verification (SciRecTrace), not the
    retrieval fusion architecture, so the fusion here is a deliberately standard, established
    design (a recognized hybridization strategy, tuned on held-out data) rather than a novel
    one -- see 05's markdown intro.

    Default weights (dense=0.6, bm25=0.1, cf=0.25, graph=0.05) were selected via a validation
    sweep (nb10) that included "dense alone" as a floor candidate; graph is kept at a small
    nonzero weight rather than the pure-ranking-optimal 0.0 so citation centrality remains a
    real, functioning signal (consistent with this system's stated design), at a small,
    measured ranking-accuracy cost. Signals are min-max normalized ACROSS THE CANDIDATE SET
    before blending -- fixing a scale-mismatch bug found during evaluation: raw dense cosine
    over this embedding space spans only ~0.76-0.87 (anisotropic embeddings), so an
    unnormalized blend let 0-1-scaled BM25/graph/CF dominate the ranking regardless of their
    nominal weight.
    """

    def __init__(self, papers_df, graph_scores: Dict,
                 lambda1: float = 0.6, lambda2: float = 0.1,
                 lambda3: float = 0.05, lambda4: float = 0.25,
                 cf_item_factors: Optional[np.ndarray] = None,
                 cf_user_factors: Optional[np.ndarray] = None,
                 cf_user_row: Optional[Dict] = None):
        self.papers_df   = papers_df
        self.graph_scores = graph_scores
        self.lambda1 = lambda1   # dense weight
        self.lambda2 = lambda2   # BM25 weight
        self.lambda3 = lambda3   # graph weight
        self.lambda4 = lambda4   # collaborative-filtering (ALS) weight
        self.cf_item_factors = cf_item_factors
        self.cf_user_factors = cf_user_factors
        self.cf_user_row = cf_user_row or {}
        self._paper_meta_cache: Dict[str, Dict] = self._build_cache()

    def _build_cache(self) -> Dict:
        cache = {}
        id_col = next((c for c in ["arxiv_id", "paper_id", "paperId", "id"] if c in self.papers_df.columns), None)
        if id_col:
            for _, row in self.papers_df.iterrows():
                cache[str(row[id_col])] = row.to_dict()
        return cache

    def _get_meta(self, paper_id: str) -> Dict:
        return self._paper_meta_cache.get(str(paper_id), {"paper_id": paper_id})

    def _cf_score(self, pid: str, user_id: Optional[int]) -> float:
        """Raw ALS dot-product score for (user_id, pid); 0.0 if either has no CF vector
        (cold-start user, or a paper outside the embedding index) -- graceful fallback to
        content+graph only for that candidate, same spirit as the base paper's RAG
        cold-start fallback."""
        if self.cf_item_factors is None or user_id is None:
            return 0.0
        urow = self.cf_user_row.get(user_id)
        if urow is None:
            return 0.0
        idx = id_mapping.get(str(pid))
        if idx is None:
            return 0.0
        return float(np.dot(self.cf_user_factors[urow], self.cf_item_factors[idx]))

    @staticmethod
    def _minmax(vals: Dict[str, float]) -> Dict[str, float]:
        if not vals:
            return {}
        v = np.array(list(vals.values()), dtype=np.float64)
        lo, hi = v.min(), v.max()
        if hi <= lo:
            return {k: 0.0 for k in vals}
        return {k: (val - lo) / (hi - lo) for k, val in vals.items()}

    def retrieve(
        self,
        query_text: str,
        user_profile_vec: Optional[np.ndarray] = None,
        user_id: Optional[int] = None,
        top_k: int = 20,
        use_s2: bool = True,
        use_web_fallback: bool = True,
    ) -> List[CandidatePaper]:
        """
        Full hybrid retrieval pipeline:
        1. Dense + BM25 + S2AG in parallel
        2. Collaborative-filtering (ALS) + graph scoring, all four signals min-max
           normalized across the candidate set before blending
        3. CRAG evaluation
        4. Web fallback for INCORRECT papers
        5. Profile reranking
        """
        # ── Stage 1: Parallel retrieval ──────────────────────────────────────
        all_scores: Dict[str, Dict] = {}  # paper_id → {dense, bm25, sources}

        with ThreadPoolExecutor(max_workers=3) as executor:
            dense_fut = executor.submit(dense_retrieve, query_text, top_k * 2)
            bm25_fut  = executor.submit(bm25_retrieve, query_text, top_k * 2)
            s2_fut    = executor.submit(s2_retrieve, query_text, top_k) if use_s2 else None

            dense_results = dense_fut.result()
            bm25_results  = bm25_fut.result()
            s2_results    = s2_fut.result() if s2_fut else []

        for pid, score in dense_results:
            if pid not in all_scores:
                all_scores[pid] = {"dense": 0, "bm25": 0, "sources": []}
            all_scores[pid]["dense"] = score
            all_scores[pid]["sources"].append("dense")

        for pid, score in bm25_results:
            if pid not in all_scores:
                all_scores[pid] = {"dense": 0, "bm25": 0, "sources": []}
            all_scores[pid]["bm25"] = score
            all_scores[pid]["sources"].append("bm25")

        for pid, score in s2_results:
            if pid not in all_scores:
                all_scores[pid] = {"dense": score * 0.6, "bm25": 0, "sources": []}
            all_scores[pid]["sources"].append("s2api")

        # ── Stage 2: CF + graph scoring, then normalize ALL FOUR signals across the
        # candidate set before blending (fixes the scale-mismatch bug -- see class docstring).
        raw_graph = {pid: float(self.graph_scores.get(str(pid), 0.0)) for pid in all_scores}
        raw_cf    = {pid: self._cf_score(pid, user_id) for pid in all_scores}
        raw_dense = {pid: s["dense"] for pid, s in all_scores.items()}
        raw_bm25  = {pid: s["bm25"] for pid, s in all_scores.items()}

        n_dense, n_bm25 = self._minmax(raw_dense), self._minmax(raw_bm25)
        n_graph, n_cf   = self._minmax(raw_graph), self._minmax(raw_cf)

        candidates = []
        for pid, scores in all_scores.items():
            g_score  = raw_graph[pid]
            cf_score = raw_cf[pid]
            final = (self.lambda1 * n_dense.get(pid, 0.0) +
                     self.lambda2 * n_bm25.get(pid, 0.0) +
                     self.lambda3 * n_graph.get(pid, 0.0) +
                     self.lambda4 * n_cf.get(pid, 0.0))

            # Profile reranking: boost if paper is similar to user profile
            if user_profile_vec is not None:
                paper_emb = None
                if str(pid) in id_mapping:
                    paper_emb = emb_matrix[id_mapping[str(pid)]]
                if paper_emb is not None:
                    profile_sim = float(
                        np.dot(user_profile_vec, paper_emb) /
                        (np.linalg.norm(user_profile_vec) * np.linalg.norm(paper_emb) + 1e-9)
                    )
                    final = 0.7 * final + 0.3 * profile_sim

            meta = self._get_meta(pid)
            crag_label = crag_evaluate(pid, query_text, scores["dense"])

            candidate = CandidatePaper(
                paper_id=pid,
                title=str(meta.get("title", "")),
                abstract=str(meta.get("abstract", ""))[:500],
                year=int(meta["year"]) if "year" in meta and pd.notna(meta.get("year")) else None,
                venue=str(meta.get("venue", "")),
                authors=str(meta.get("authors", "")),
                dense_score=scores["dense"],
                bm25_score=scores["bm25"],
                graph_score=g_score,
                final_score=final,
                sources=scores["sources"],
                retrieval_confidence=crag_label,
            )
            candidate.cf_score = cf_score
            candidates.append(candidate)

        # ── Stage 3: CRAG — handle INCORRECT/AMBIGUOUS ───────────────────────
        correct = [c for c in candidates if c.retrieval_confidence == "CORRECT"]
        incorrect = [c for c in candidates if c.retrieval_confidence == "INCORRECT"]

        # If too few CORRECT results and web fallback enabled, trigger Tavily
        if use_web_fallback and len(correct) < top_k // 2:
            web_results = tavily_retrieve(query_text, top_k=5)
            for wurl, wscore in web_results:
                # Web results are stubs — enrich in production via DOI resolution
                web_candidate = CandidatePaper(
                    paper_id=wurl,
                    title=f"[Web] {wurl[:80]}",
                    dense_score=wscore * 0.5,
                    final_score=wscore * 0.4,
                    sources=["web"],
                    retrieval_confidence="AMBIGUOUS",
                )
                correct.append(web_candidate)

        # Sort by final score and return top_k
        all_sorted = sorted(correct + [
            c for c in candidates if c.retrieval_confidence == "AMBIGUOUS"
        ], key=lambda c: c.final_score, reverse=True)

        return all_sorted[:top_k]


print("HybridRetrievalAgent defined (dense + BM25 + CF + graph, shortlist-normalized fusion).")

## 5. End-to-End Retrieval Test

In [ ]:
retrieval_agent = HybridRetrievalAgent(
    papers_df, graph_scores,
    cf_item_factors=cf_item_factors, cf_user_factors=cf_user_factors, cf_user_row=cf_user_row,
)

# Select a sample user (graceful if no profiles attached)
if user_profiles:
    sample_uid = list(user_profiles.keys())[0]
    _raw = user_profiles[sample_uid]
    profile_vec_list = _raw.get("profile_vector", []) if isinstance(_raw, dict) else _raw
    profile_vec = np.array(profile_vec_list, dtype=np.float32) if profile_vec_list else None
    print(f"personalizing for user {sample_uid}")
else:
    profile_vec = None
    print("no user profiles -> running without personalization")

# NOTE: user_profiles' own keys ("0", "1", ...) do NOT correspond to rated_papers.csv /
# als_user_ids.json user_ids (a separate, pre-existing mismatch -- see nb10's notes), so a CF
# vector can't be looked up for `sample_uid` above. Demo the CF path with a real user_id from
# the shared ALS model directly instead, so this end-to-end test actually exercises it.
demo_cf_user_id = next(iter(cf_user_row), None) if cf_user_row else None

test_query = "hallucination detection in scientific paper recommendation using LLMs"

results = retrieval_agent.retrieve(
    query_text=test_query,
    user_profile_vec=profile_vec,
    user_id=demo_cf_user_id,
    top_k=10,
    use_s2=False,            # corpus-grounded: external S2/web ids aren't in the corpus,
    use_web_fallback=False,  # can't be graph/profile-enriched, and won't match GT arxiv_ids
)

print(f"Query: {test_query}")
print(f"CF user_id used: {demo_cf_user_id}\n")
print(f"Retrieved {len(results)} candidates:\n")
for r in results:
    print(f"  [{r.final_score:.3f}] {r.title[:60]} | CF={r.cf_score:.3f} | "
          f"CRAG={r.retrieval_confidence} | src={r.sources}")

## 6. Retrieval Statistics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Source distribution
from collections import Counter
source_counts = Counter(src for r in results for src in r.sources)
crag_counts   = Counter(r.retrieval_confidence for r in results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(source_counts.keys(), source_counts.values(), color="steelblue")
axes[0].set_title("Retrieval Source Distribution")
axes[0].set_ylabel("Count")

axes[1].bar(crag_counts.keys(), crag_counts.values(),
             color=["green", "orange", "red"][:len(crag_counts)])
axes[1].set_title("CRAG Confidence Labels")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(OUT_DIR / "fig_retrieval_stats.png")
plt.show()

## 7. Summary

| Feature | Implementation |
|---|---|
| Dense retrieval | GTE-Large embeddings, direct cosine over the in-memory matrix (no ChromaDB) |
| Lexical retrieval | Okapi BM25 |
| Collaborative filtering | Implicit-feedback ALS (shared model, trained once in nb10) → `lambda4` |
| Graph scoring | Citation-graph centrality (S2AG `citationCount`, log-scaled) → `lambda3` |
| Fusion | Dense/BM25/CF/graph min-max normalized per candidate set, then weighted sum (Burke, 2002) |
| Profile reranking | Cosine similarity of candidate to user profile vector (nb04) |
| External fallback (optional, off by default) | S2 paper search + Tavily web (external ids; augmentation only) |
| CRAG labels | CORRECT / AMBIGUOUS / INCORRECT per candidate |

**Next**: Notebook 06 — Critic-Generator Agent